# Lesson 6 | What is RTL?

We now need to state ports, stored state, and edge-triggered updates explicitly. Today asks:
> **How can code-like text describe digital hardware rather than sequential CPU instructions?**

Primary concept: **Register-Transfer Level (RTL)**.


## 1. Four terms

**Hardware Description Language (HDL):** a language class for digital hardware structure/behavior.

**SystemVerilog:** the HDL and verification language used here.

**RTL:** focuses on what registers store and how data is computed/transferred across clock cycles.

**module / port:** a module is a bounded hardware unit; ports are input/output signals crossing its boundary.


## 2. HDL can look like software while meaning something different

Adjacent Python lines usually imply sequence. RTL often describes hardware relationships existing at the same time. Ask about inputs, outputs, state, and clock edge before asking about line order.


## 3. Minimal RTL: clocked accumulator

```systemverilog
module clocked_accumulator #(
    parameter int WIDTH = 8
) (
    input  logic clk,
    input  logic rst_n,
    input  logic signed [WIDTH-1:0] input_value,
    output logic signed [WIDTH-1:0] state
);
    always_ff @(posedge clk) begin
        if (!rst_n) state <= '0;
        else        state <= state + input_value;
    end
endmodule
```


## 4. Read it line by line

`input/output` declare port direction; `logic` is a common signal type; `[WIDTH-1:0]` is width.

`always_ff @(posedge clk)` describes register update on a rising edge. `<=` is a **nonblocking assignment**, commonly used for sequential RTL.


## 5. Reset is part of the contract

The `_n` suffix in `rst_n` indicates active-low. This module uses synchronous reset: `rst_n=0` takes effect on a clock edge. Reset polarity and timing must be explicit.


## 6. Run

The next cell looks for **Icarus Verilog**. It is a system HDL simulator, not a `uv` Python dependency. Missing tools are reported as “simulation did not run.”


In [ ]:
from pathlib import Path
import shutil, subprocess, tempfile

def repo_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p/'pyproject.toml').exists() and (p/'lessons').exists(): return p
    raise FileNotFoundError('Run inside FPGA-FlyBrain')

root = repo_root(); iverilog = shutil.which('iverilog'); vvp = shutil.which('vvp')
if not (iverilog and vvp):
    print('Icarus Verilog not found; simulation did not run.')
else:
    with tempfile.TemporaryDirectory() as td:
        out = Path(td)/'l6.out'
        subprocess.run([iverilog,'-g2012','-o',str(out),str(root/'rtl/learning/clocked_accumulator.sv'),str(root/'tb/learning/clocked_accumulator_tb.sv')],check=True)
        r=subprocess.run([vvp,str(out)],check=True,text=True,capture_output=True)
        print(r.stdout.strip())


## 7. Observe / Try It

The testbench applies `1,2,3` and checks post-edge state `1,3,6`; success prints `PASS lesson06 clocked_accumulator`.

Work through `[2,-1,4]` by hand, then add those checks yourself.


## 8. AI Task / Human Check

Ask AI only to annotate module, parameter, ports, state, and clocked update in the existing RTL.

Without AI, explain HDL vs software semantics, module/port, stored state, `posedge`, and why reset belongs in the contract.


## 9. Engineering Handoff / Project Trace

`rtl/learning/clocked_accumulator.sv` and its testbench are teaching artifacts, not formal `MOD-003`.

- Lesson: `LSN-006`
- Prepares: `RMD-004`
- RTL: `rtl/learning/clocked_accumulator.sv`
- Check: `tb/learning/clocked_accumulator_tb.sv`


## 10. Exit Ticket

You can explain RTL, HDL, SystemVerilog, module, and port, then identify inputs, outputs, state, and clocked update in a small module.
